# URL Phishing Detection

This notebook implements the complete machine learning lifecycle for **URL Phishing Detection** using the **PhiUSIIL Phishing URL Dataset**.

### Workflow Overview:
- **1. Imports**: Standard and scientific machine learning libraries.
- **2. Dataset Loading**: Ingestion of raw URL data.
- **3. Dataset Inspection**: Comprehensive shape, schema, missing value, and label verification.
- **4. Data Cleaning**: In-memory deduplication and canonical target mapping.
- **5. Train-Test Split**: Stratified 80/20 split executed prior to feature engineering.
- **6. URL Feature Engineering**: 23 offline lexical and structural URL features (no network calls).
- **7. Feature Inspection**: Statistical summary and target correlation analysis.
- **8. Model Training**: Training Logistic Regression, Random Forest, and XGBoost.
- **9. Model Evaluation**: Comprehensive metrics on unseen test data.
- **10. Model Comparison**: Empirical comparison table and confusion matrices.
- **11. Best Model Selection**: Evidence-based model selection.
- **12. Error Analysis**: False Positive and False Negative diagnostic review.
- **13. Final Pipeline Training/Saving**: End-to-end scikit-learn Pipeline serialization.
- **14. Reload Verification**: Real test sample inference check on deserialized model.
- **15. Final Summary**: Consolidated execution summary.


## 1. Imports

Importing required core libraries for data handling, feature extraction, model training, evaluation, and pipeline serialization.

In [1]:
import os
import re
import time
import json
import joblib
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print("All dependencies imported successfully.")


All dependencies imported successfully.


## 2. Dataset Loading

Loading the raw dataset `PhiUSIIL_Phishing_URL_Dataset.csv` from `ml/data/raw/`.

In [2]:
csv_path = Path("c:/Users/baps/OneDrive/Desktop/ai-scam-phishing-detector/ml/data/raw/PhiUSIIL_Phishing_URL_Dataset.csv")

t0 = time.time()
df_raw = pd.read_csv(csv_path)
load_time = time.time() - t0

print(f"Dataset loaded in {load_time:.2f} seconds.")
print(f"Dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")


Dataset loaded in 1.45 seconds.
Dataset shape: 235,795 rows x 56 columns


## 3. Dataset Inspection

Inspecting dataset shape, column names, data types, missing values, duplicates, and analyzing target labels.

In [3]:
# 1. Dataset overview
print("=== DATASET INFORMATION ===")
print(f"Total Rows:    {df_raw.shape[0]:,}")
print(f"Total Columns: {df_raw.shape[1]}")
print("\nData Types Summary:")
print(df_raw.dtypes.value_counts())

# 2. Check missing values
null_counts = df_raw.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]
print(f"\nColumns with Missing Values: {len(cols_with_nulls)}")
if len(cols_with_nulls) > 0:
    print(cols_with_nulls)
else:
    print("Zero missing values across all 56 columns.")

# 3. Check duplicate rows
full_dups = df_raw.duplicated().sum()
url_dups = df_raw['URL'].duplicated().sum()
print(f"\nFull row duplicates: {full_dups}")
print(f"Duplicate URLs:      {url_dups} (Unique URLs: {df_raw['URL'].nunique():,})")

# 4. Display First 5 and Last 5 URLs
print("\n=== FIRST 5 ROWS (URL & label) ===")
print(df_raw[['URL', 'label']].head(5).to_string())

print("\n=== LAST 5 ROWS (URL & label) ===")
print(df_raw[['URL', 'label']].tail(5).to_string())

# 5. Label Distribution and Ground Truth Analysis
raw_label_counts = df_raw['label'].value_counts()
print("\n=== RAW TARGET DISTRIBUTION ===")
print("Raw Label '1':", f"{raw_label_counts.get(1, 0):,} ({raw_label_counts.get(1, 0)/len(df_raw)*100:.2f}%)")
print("Raw Label '0':", f"{raw_label_counts.get(0, 0):,} ({raw_label_counts.get(0, 0)/len(df_raw)*100:.2f}%)")

print("\nSample URLs for raw label 1:")
for u in df_raw[df_raw['label'] == 1]['URL'].head(5):
    print("  [1] ", u)

print("\nSample URLs for raw label 0:")
for u in df_raw[df_raw['label'] == 0]['URL'].head(5):
    print("  [0] ", u)


=== DATASET INFORMATION ===
Total Rows:    235,795
Total Columns: 56

Data Types Summary:
int64      41
float64    10
str         5
Name: count, dtype: int64

Columns with Missing Values: 0
Zero missing values across all 56 columns.



Full row duplicates: 0
Duplicate URLs:      425 (Unique URLs: 235,370)

=== FIRST 5 ROWS (URL & label) ===
                                  URL  label
0    https://www.southbankmosaics.com      1
1            https://www.uni-mainz.de      1
2      https://www.voicefmradio.co.uk      1
3         https://www.sfnmjournal.com      1
4  https://www.rewildingargentina.org      1

=== LAST 5 ROWS (URL & label) ===
                                                             URL  label
235790                            https://www.skincareliving.com      1
235791                             https://www.winchester.gov.uk      1
235792                           https://www.nononsensedesign.be      1
235793  https://patient-cell-40f5.updatedlogmylogin.workers.dev/      0
235794                        https://www.alternativefinland.com      1

=== RAW TARGET DISTRIBUTION ===
Raw Label '1': 134,850 (57.19%)
Raw Label '0': 100,945 (42.81%)

Sample URLs for raw label 1:
  [1]  https://www.southbank

### Label Identification & Canonical Mapping
In the published **PhiUSIIL** benchmark dataset:
- `label = 1` denotes **Safe / Legitimate URLs** (e.g., verified domains such as universities, registered radios, NGOs).
- `label = 0` denotes **Phishing / Malicious URLs** (e.g., suspicious IPFS links, dynamic DNS, free domain extensions like `.gq`, AT&T phishing lures on Weebly, Firebase hosting kits).

Per project specifications:
- We treat **Phishing** as the **positive class** (`label = 1`).
- We treat **Safe / Legitimate** as the **negative class** (`label = 0`).
- Therefore, in our cleaned target variable `y`:
  - `y = 1`: **Phishing / Scam URL** (raw label 0)
  - `y = 0`: **Safe / Legitimate URL** (raw label 1)


## 4. Data Cleaning

Cleaning the dataset in memory only (without altering the original CSV file):
1. Deduplicating on the `URL` column.
2. Verifying and dropping any empty or whitespace URLs.
3. Encoding the canonical target: `1 = Phishing`, `0 = Legitimate`.

In [4]:
# Clean in-memory
initial_rows = len(df_raw)

# Drop duplicate URLs
df_cleaned = df_raw.drop_duplicates(subset=['URL']).copy()
dups_removed = initial_rows - len(df_cleaned)

# Remove empty/null/whitespace URLs
df_cleaned = df_cleaned.dropna(subset=['URL'])
df_cleaned = df_cleaned[df_cleaned['URL'].astype(str).str.strip() != '']
empty_removed = (initial_rows - dups_removed) - len(df_cleaned)

# Map canonical target: 1 = Phishing, 0 = Safe
y_cleaned = (df_cleaned['label'] == 0).astype(int)
X_urls_cleaned = df_cleaned['URL'].astype(str).str.strip()

print("=== DATA CLEANING SUMMARY ===")
print(f"Original Row Count:          {initial_rows:,}")
print(f"Duplicate URLs Removed:      {dups_removed:,}")
print(f"Empty/Whitespace Removed:    {empty_removed:,}")
print(f"Final Cleaned Dataset Size:  {len(df_cleaned):,} rows")

class_counts = y_cleaned.value_counts()
class_pcts = y_cleaned.value_counts(normalize=True) * 100

print("\n=== FINAL TARGET CLASS DISTRIBUTION ===")
print(f"Class 0 (Safe / Legitimate): {class_counts[0]:,} ({class_pcts[0]:.2f}%)")
print(f"Class 1 (Phishing / Scam):   {class_counts[1]:,} ({class_pcts[1]:.2f}%)")


=== DATA CLEANING SUMMARY ===
Original Row Count:          235,795
Duplicate URLs Removed:      425
Empty/Whitespace Removed:    0
Final Cleaned Dataset Size:  235,370 rows

=== FINAL TARGET CLASS DISTRIBUTION ===
Class 0 (Safe / Legitimate): 134,850 (57.29%)
Class 1 (Phishing / Scam):   100,520 (42.71%)


## 5. Train-Test Split

Splitting the dataset into an **80% Training set** and a **20% Testing set** with stratification before feature extraction to prevent data leakage.

In [5]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_urls_cleaned,
    y_cleaned,
    test_size=0.20,
    random_state=42,
    stratify=y_cleaned
)

train_counts = y_train.value_counts()
test_counts = y_test.value_counts()

split_summary = pd.DataFrame({
    'Subset': ['Training Set (X_train)', 'Testing Set (X_test)'],
    'Total Samples': [f"{len(X_train_raw):,}", f"{len(X_test_raw):,}"],
    'Safe (0) Count': [f"{train_counts[0]:,}", f"{test_counts[0]:,}"],
    'Safe (0) %': [f"{(train_counts[0] / len(X_train_raw) * 100):.2f}%", f"{(test_counts[0] / len(X_test_raw) * 100):.2f}%"],
    'Phishing (1) Count': [f"{train_counts[1]:,}", f"{test_counts[1]:,}"],
    'Phishing (1) %': [f"{(train_counts[1] / len(X_train_raw) * 100):.2f}%", f"{(test_counts[1] / len(X_test_raw) * 100):.2f}%"]
})

print("=== STRATIFIED TRAIN / TEST SPLIT SUMMARY ===")
print(split_summary.to_string(index=False))

# Verify representation
assert set(y_train.unique()) == {0, 1}, "y_train missing classes"
assert set(y_test.unique()) == {0, 1}, "y_test missing classes"
print("\nVerification Passed: Both classes are fully represented with identical proportions.")


=== STRATIFIED TRAIN / TEST SPLIT SUMMARY ===
                Subset Total Samples Safe (0) Count Safe (0) % Phishing (1) Count Phishing (1) %
Training Set (X_train)       188,296        107,880     57.29%             80,416         42.71%
  Testing Set (X_test)        47,074         26,970     57.29%             20,104         42.71%

Verification Passed: Both classes are fully represented with identical proportions.


## 6. URL Feature Engineering

We extract 23 domain-specific lexical and structural features directly from the raw URL strings.

### Safety & Offline Protocol:
- **No network requests**: The model will never ping, resolve, or fetch remote servers.
- **Pure Lexical Analysis**: Features are calculated purely from the character distribution, URL grammar, and domain syntax.

### Extracted Feature Set (23 Features):
1. `url_len`: Overall length of the URL string.
2. `hostname_len`: Length of the network location (FQDN).
3. `path_len`: Length of the path component.
4. `query_len`: Length of the query string.
5. `dot_count`: Frequency of '.' delimiters.
6. `hyphen_count`: Frequency of '-' characters.
7. `underscore_count`: Frequency of '_' characters.
8. `slash_count`: Frequency of '/' path separators.
9. `question_count`: Frequency of '?' query indicators.
10. `equal_count`: Frequency of '=' parameter operators.
11. `ampersand_count`: Frequency of '&' query separators.
12. `at_count`: Presence of '@' credential symbol.
13. `digit_count`: Total numeric characters [0-9].
14. `letter_count`: Total alphabetic characters [a-zA-Z].
15. `special_count`: Total non-alphanumeric special characters.
16. `digit_ratio`: Proportion of digits to URL length.
17. `special_ratio`: Proportion of special characters to URL length.
18. `no_of_subdomains`: Number of subdomain tiers.
19. `is_ip`: Flag indicating if hostname is an IPv4 address.
20. `is_https`: Flag indicating HTTPS scheme.
21. `is_shortener`: Flag indicating recognized URL shortener domain.
22. `keyword_count`: Frequency of sensitive security/financial keywords.
23. `tld_len`: Character length of top-level domain.


In [6]:
SHORTENERS = {
    'bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'is.gd', 'buff.ly',
    'adf.ly', 'bit.do', 'cutt.ly', 'rb.gy', 'shorte.st', 'tiny.cc'
}

SUSPICIOUS_KEYWORDS = [
    'login', 'verify', 'update', 'secure', 'banking', 'account', 'signin',
    'confirm', 'security', 'wallet', 'admin', 'service', 'support', 'password'
]

IP_PATTERN = re.compile(r'^(?:http[s]?://)?(?:[0-9]{1,3}\.){3}[0-9]{1,3}(?::[0-9]+)?(?:/.*)?$')

def extract_url_features_single(url: str):
    url_str = str(url).strip()
    url_len = len(url_str)
    
    # Ensure scheme for reliable urlparse
    if not (url_str.startswith('http://') or url_str.startswith('https://')):
        parse_target = 'http://' + url_str
    else:
        parse_target = url_str
        
    try:
        parsed = urlparse(parse_target)
        netloc = parsed.netloc.lower()
        path = parsed.path
        query = parsed.query
    except Exception:
        netloc = ""
        path = ""
        query = ""

    hostname = netloc.split(':')[0] if ':' in netloc else netloc
    
    # Character counts
    dot_count = url_str.count('.')
    hyphen_count = url_str.count('-')
    underscore_count = url_str.count('_')
    slash_count = url_str.count('/')
    question_count = url_str.count('?')
    equal_count = url_str.count('=')
    ampersand_count = url_str.count('&')
    at_count = url_str.count('@')
    
    digit_count = sum(c.isdigit() for c in url_str)
    letter_count = sum(c.isalpha() for c in url_str)
    special_count = sum(not c.isalnum() for c in url_str)
    
    digit_ratio = digit_count / url_len if url_len > 0 else 0.0
    special_ratio = special_count / url_len if url_len > 0 else 0.0
    
    # Hostname & path features
    hostname_len = len(hostname)
    path_len = len(path)
    query_len = len(query)
    
    # Subdomain count
    parts = hostname.split('.')
    no_of_subdomains = max(0, len(parts) - 2) if len(parts) >= 2 else 0
    
    # IP check
    is_ip = 1 if IP_PATTERN.match(url_str) or re.match(r'^(?:[0-9]{1,3}\.){3}[0-9]{1,3}$', hostname) else 0
    
    # HTTPS check
    is_https = 1 if url_str.lower().startswith('https://') else 0
    
    # Shortener check
    is_shortener = 1 if hostname in SHORTENERS else 0
    
    # Suspicious keywords
    url_lower = url_str.lower()
    keyword_count = sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in url_lower)
    
    # TLD length
    tld = parts[-1] if len(parts) > 1 else ""
    tld_len = len(tld)
    
    return [
        url_len, hostname_len, path_len, query_len,
        dot_count, hyphen_count, underscore_count, slash_count,
        question_count, equal_count, ampersand_count, at_count,
        digit_count, letter_count, special_count,
        digit_ratio, special_ratio, no_of_subdomains,
        is_ip, is_https, is_shortener, keyword_count, tld_len
    ]

FEATURE_NAMES = [
    'url_len', 'hostname_len', 'path_len', 'query_len',
    'dot_count', 'hyphen_count', 'underscore_count', 'slash_count',
    'question_count', 'equal_count', 'ampersand_count', 'at_count',
    'digit_count', 'letter_count', 'special_count',
    'digit_ratio', 'special_ratio', 'no_of_subdomains',
    'is_ip', 'is_https', 'is_shortener', 'keyword_count', 'tld_len'
]

print(f"Defined {len(FEATURE_NAMES)} lexical URL features.")


Defined 23 lexical URL features.


In [7]:
print("Extracting features for training set (X_train)...")
t0 = time.time()
X_train_feats = np.array([extract_url_features_single(u) for u in X_train_raw], dtype=np.float32)
t_train_fe = time.time() - t0
print(f"X_train features extracted in {t_train_fe:.2f}s: shape = {X_train_feats.shape}")

print("Extracting features for testing set (X_test)...")
t0 = time.time()
X_test_feats = np.array([extract_url_features_single(u) for u in X_test_raw], dtype=np.float32)
t_test_fe = time.time() - t0
print(f"X_test features extracted in {t_test_fe:.2f}s: shape = {X_test_feats.shape}")


Extracting features for training set (X_train)...


X_train features extracted in 3.19s: shape = (188296, 23)
Extracting features for testing set (X_test)...


X_test features extracted in 0.84s: shape = (47074, 23)


## 7. Feature Inspection

Statistical summary of extracted features and Pearson correlation analysis against the target `is_phishing`.

In [8]:
df_feats_train = pd.DataFrame(X_train_feats, columns=FEATURE_NAMES)
df_feats_train['is_phishing'] = y_train.values

print("=== SUMMARY STATISTICS (FIRST 10 FEATURES) ===")
print(df_feats_train[FEATURE_NAMES[:10]].describe().round(2).to_string())

# Calculate correlations with phishing
corrs = df_feats_train[FEATURE_NAMES].apply(lambda col: col.corr(df_feats_train['is_phishing']))
corrs_sorted = corrs.sort_values(ascending=False)

print("\n=== TOP 10 FEATURES CORRELATED WITH PHISHING ===")
for feat, corr_val in corrs_sorted.head(10).items():
    print(f"  {feat:<20}: {corr_val:+.4f}")


=== SUMMARY STATISTICS (FIRST 10 FEATURES) ===
         url_len  hostname_len   path_len  query_len  dot_count  hyphen_count  underscore_count  slash_count  question_count  equal_count
count  188296.00     188296.00  188296.00  188296.00  188296.00     188296.00         188296.00    188296.00       188296.00    188296.00
mean       35.41         21.47       3.73       2.24       2.26          0.35              0.04         2.43            0.03         0.06
std        43.56          9.15      25.26      25.53       0.94          1.64              0.59         1.05            0.19         1.01
min        14.00          4.00       0.00       0.00       1.00          0.00              0.00         2.00            0.00         0.00
25%        24.00         16.00       0.00       0.00       2.00          0.00              0.00         2.00            0.00         0.00
50%        28.00         20.00       0.00       0.00       2.00          0.00              0.00         2.00            0.00 


=== TOP 10 FEATURES CORRELATED WITH PHISHING ===
  slash_count         : +0.4799
  digit_ratio         : +0.4336
  hostname_len        : +0.2830
  special_count       : +0.2305
  url_len             : +0.2174
  letter_count        : +0.2028
  keyword_count       : +0.1954
  hyphen_count        : +0.1902
  question_count      : +0.1759
  path_len            : +0.1712


## 8. Model Training

Training three diverse classifiers:
1. **Logistic Regression** (Standardized baseline)
2. **Random Forest** (`n_estimators=100`, `max_depth=20`)
3. **XGBoost** (`n_estimators=150`, `max_depth=8`, `learning_rate=0.1`)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feats)
X_test_scaled = scaler.transform(X_test_feats)

models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Random Forest': (RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1), False),
    'XGBoost': (XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss'), False)
}

trained_models = {}
fit_times = {}

for name, (model, use_scaled) in models.items():
    print(f"Training {name} on {X_train_feats.shape[0]:,} samples...")
    t_start = time.time()
    X_tr = X_train_scaled if use_scaled else X_train_feats
    model.fit(X_tr, y_train)
    duration = time.time() - t_start
    trained_models[name] = (model, use_scaled)
    fit_times[name] = duration
    print(f"  -> {name} trained in {duration:.2f}s")


Training Logistic Regression on 188,296 samples...


  -> Logistic Regression trained in 0.15s
Training Random Forest on 188,296 samples...


  -> Random Forest trained in 8.05s
Training XGBoost on 188,296 samples...


  -> XGBoost trained in 0.68s


## 9. Model Evaluation

Evaluating all models on the untouched testing set (`X_test_feats`, `y_test`). The phishing class is treated as positive (`label=1`).

In [10]:
results_list = []
confusion_matrices = {}

for name, (model, use_scaled) in trained_models.items():
    X_te = X_test_scaled if use_scaled else X_test_feats
    
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    
    acc = float(accuracy_score(y_test, y_pred))
    prec = float(precision_score(y_test, y_pred))
    rec = float(recall_score(y_test, y_pred))
    f1 = float(f1_score(y_test, y_pred))
    auc = float(roc_auc_score(y_test, y_proba))
    cm = confusion_matrix(y_test, y_pred)
    
    confusion_matrices[name] = cm
    results_list.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc,
        'Fit Time (s)': fit_times[name]
    })

print("All models evaluated successfully.")


All models evaluated successfully.


## 10. Model Comparison

Comparative summary of accuracy, precision, recall, F1, ROC-AUC, and confusion matrix diagnostics.

In [11]:
df_comparison = pd.DataFrame(results_list)
print("=== URL PHISHING MODEL COMPARISON ===")
print(df_comparison.to_string(index=False))

print("\n=== CONFUSION MATRICES (TEST SET: 47,074 SAMPLES) ===")
for name, cm in confusion_matrices.items():
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) * 100
    fnr = fn / (fn + tp) * 100
    print(f"\nModel: {name}")
    print(f"  True Negatives (Safe correctly identified):     {tn:>6,}")
    print(f"  False Positives (Safe flagged as Phishing):     {fp:>6,} (FPR: {fpr:.2f}%)")
    print(f"  False Negatives (Phishing missed as Safe):      {fn:>6,} (FNR: {fnr:.2f}%)")
    print(f"  True Positives (Phishing correctly caught):     {tp:>6,}")


=== URL PHISHING MODEL COMPARISON ===
              Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC  Fit Time (s)
Logistic Regression  0.993839   0.999345 0.986222  0.992740 0.996080      0.147213
      Random Forest  0.995581   0.998197 0.991444  0.994809 0.997396      8.053295
            XGBoost  0.995900   0.999098 0.991295  0.995181 0.997914      0.675045

=== CONFUSION MATRICES (TEST SET: 47,074 SAMPLES) ===

Model: Logistic Regression
  True Negatives (Safe correctly identified):     26,957
  False Positives (Safe flagged as Phishing):         13 (FPR: 0.05%)
  False Negatives (Phishing missed as Safe):         277 (FNR: 1.38%)
  True Positives (Phishing correctly caught):     19,827

Model: Random Forest
  True Negatives (Safe correctly identified):     26,934
  False Positives (Safe flagged as Phishing):         36 (FPR: 0.13%)
  False Negatives (Phishing missed as Safe):         172 (FNR: 0.86%)
  True Positives (Phishing correctly caught):     19,932

Model: XGBoost
 

## 11. Best Model Selection

Selecting the optimal model based on empirical test metrics.

In [12]:
best_model_name = "XGBoost"
best_model, best_uses_scaled = trained_models[best_model_name]

print(f"Selected Best Model: {best_model_name}")
print(f"  Accuracy:  {df_comparison.loc[df_comparison['Model'] == best_model_name, 'Accuracy'].values[0]:.4f}")
print(f"  Precision: {df_comparison.loc[df_comparison['Model'] == best_model_name, 'Precision'].values[0]:.4f}")
print(f"  Recall:    {df_comparison.loc[df_comparison['Model'] == best_model_name, 'Recall'].values[0]:.4f}")
print(f"  F1-Score:  {df_comparison.loc[df_comparison['Model'] == best_model_name, 'F1-Score'].values[0]:.4f}")
print(f"  ROC-AUC:   {df_comparison.loc[df_comparison['Model'] == best_model_name, 'ROC-AUC'].values[0]:.4f}")
print("\nRationale:")
print("- XGBoost achieved the highest F1-Score (0.9952), Accuracy (99.59%), and ROC-AUC (0.9979).")
print("- It exhibits exceptional precision (99.91%), resulting in only 18 False Positives across 26,970 legitimate test URLs.")
print("- Fast inference and sub-second training speed make it ideal for high-throughput URL analysis.")


Selected Best Model: XGBoost
  Accuracy:  0.9959
  Precision: 0.9991
  Recall:    0.9913
  F1-Score:  0.9952
  ROC-AUC:   0.9979

Rationale:
- XGBoost achieved the highest F1-Score (0.9952), Accuracy (99.59%), and ROC-AUC (0.9979).
- It exhibits exceptional precision (99.91%), resulting in only 18 False Positives across 26,970 legitimate test URLs.
- Fast inference and sub-second training speed make it ideal for high-throughput URL analysis.


## 12. Error Analysis

Deep-dive inspection into False Positives and False Negatives produced by the best model on the test set.

In [13]:
y_pred_best = best_model.predict(X_test_feats)
y_proba_best = best_model.predict_proba(X_test_feats)[:, 1]

# Identify misclassified samples
fp_mask = (y_test.values == 0) & (y_pred_best == 1)
fn_mask = (y_test.values == 1) & (y_pred_best == 0)

fp_indices = np.where(fp_mask)[0]
fn_indices = np.where(fn_mask)[0]

print(f"=== ERROR ANALYSIS (TEST SET N = {len(y_test):,}) ===")
print(f"Total False Positives: {len(fp_indices):,} / {sum(y_test == 0):,} safe URLs (FPR: {len(fp_indices)/sum(y_test == 0)*100:.3f}%)")
print(f"Total False Negatives: {len(fn_indices):,} / {sum(y_test == 1):,} phishing URLs (FNR: {len(fn_indices)/sum(y_test == 1)*100:.3f}%)")

print("\n--- REPRESENTATIVE FALSE POSITIVES (Safe URLs predicted as Phishing) ---")
for idx in fp_indices[:5]:
    url = X_test_raw.iloc[idx]
    score = y_proba_best[idx]
    print(f"  Predicted Phish Prob: {score:.4f} | URL: {url[:100]}")

print("\n--- REPRESENTATIVE FALSE NEGATIVES (Phishing URLs predicted as Safe) ---")
for idx in fn_indices[:5]:
    url = X_test_raw.iloc[idx]
    score = y_proba_best[idx]
    print(f"  Predicted Phish Prob: {score:.4f} | URL: {url[:100]}")


=== ERROR ANALYSIS (TEST SET N = 47,074) ===
Total False Positives: 18 / 26,970 safe URLs (FPR: 0.067%)
Total False Negatives: 175 / 20,104 phishing URLs (FNR: 0.870%)

--- REPRESENTATIVE FALSE POSITIVES (Safe URLs predicted as Phishing) ---
  Predicted Phish Prob: 0.7669 | URL: https://www.clwydianrangeanddeevalleyaonb.org.uk
  Predicted Phish Prob: 0.6944 | URL: https://www.wirtschaftsfoerderung-dortmund.de
  Predicted Phish Prob: 0.5573 | URL: https://www.business-services.upenn.edu
  Predicted Phish Prob: 0.8720 | URL: https://www.180360.com
  Predicted Phish Prob: 0.5045 | URL: https://www.check24-partnerprogramm.de

--- REPRESENTATIVE FALSE NEGATIVES (Phishing URLs predicted as Safe) ---
  Predicted Phish Prob: 0.0031 | URL: https://what.promerc.repl.co
  Predicted Phish Prob: 0.0023 | URL: https://www.cfg.me
  Predicted Phish Prob: 0.0021 | URL: https://s.smcbacmzu.icu
  Predicted Phish Prob: 0.0142 | URL: https://www.michelonturismo.com.br
  Predicted Phish Prob: 0.0315 | URL: 

## 13. Final Pipeline Training/Saving

Packaging feature engineering and XGBoost into an end-to-end `scikit-learn Pipeline` for atomic inference.
Serializing pipeline to `ml/models/url_phishing_pipeline.joblib` and factual metadata to `ml/models/url_phishing_metadata.json`.


In [14]:
class URLFeatureExtractor(BaseEstimator, TransformerMixin):
    """
    Custom scikit-learn Transformer for extracting lexical and structural URL features.
    Enables atomic end-to-end inference from raw URL strings.
    """
    def __init__(self):
        self.feature_names_ = FEATURE_NAMES
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        if isinstance(X, str):
            X = [X]
        features = [extract_url_features_single(url) for url in X]
        return np.array(features, dtype=np.float32)

# Build pipeline
url_pipeline = Pipeline([
    ('feature_extractor', URLFeatureExtractor()),
    ('classifier', XGBClassifier(
        n_estimators=150,
        max_depth=8,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    ))
])

print("Fitting complete URL phishing pipeline on X_train_raw...")
t0 = time.time()
url_pipeline.fit(X_train_raw, y_train)
print(f"Pipeline fitted in {time.time() - t0:.2f}s.")

# Verify pipeline predictions match standalone best model
y_pred_pipe = url_pipeline.predict(X_test_raw)
np.testing.assert_array_equal(y_pred_pipe, y_pred_best)
print("Verification Passed: Pipeline predictions match standalone XGBoost predictions identically.")


Fitting complete URL phishing pipeline on X_train_raw...


Pipeline fitted in 4.44s.


Verification Passed: Pipeline predictions match standalone XGBoost predictions identically.


In [15]:
# Create models directory if necessary
models_dir = Path("c:/Users/baps/OneDrive/Desktop/ai-scam-phishing-detector/ml/models")
models_dir.mkdir(parents=True, exist_ok=True)

pipeline_path = models_dir / "url_phishing_pipeline.joblib"
metadata_path = models_dir / "url_phishing_metadata.json"

# Save pipeline artifact
joblib.dump(url_pipeline, pipeline_path)

# Prepare factual metadata
cm_best = confusion_matrices['XGBoost']
tn_b, fp_b, fn_b, tp_b = [int(v) for v in cm_best.ravel()]

metadata = {
    "model_name": "XGBoost Classifier (XGBClassifier)",
    "pipeline_steps": ["feature_extractor", "classifier"],
    "task": "binary_url_phishing_classification",
    "input_type": "raw_url_string",
    "target_labels": {
        "0": "Safe / Legitimate URL",
        "1": "Phishing / Scam URL"
    },
    "positive_label": 1,
    "negative_label": 0,
    "feature_engineering": {
        "feature_count": len(FEATURE_NAMES),
        "feature_names": FEATURE_NAMES,
        "type": "lexical_and_structural_offline_extraction",
        "network_requests": False
    },
    "model_parameters": {
        "n_estimators": 150,
        "max_depth": 8,
        "learning_rate": 0.1,
        "random_state": 42,
        "eval_metric": "logloss"
    },
    "dataset_summary": {
        "dataset_name": "PhiUSIIL_Phishing_URL_Dataset",
        "cleaned_samples": int(len(df_cleaned)),
        "train_samples": int(len(X_train_raw)),
        "test_samples": int(len(X_test_raw)),
        "stratified": True,
        "test_size": 0.20,
        "random_state": 42
    },
    "class_distribution": {
        "safe_samples": int(sum(y_cleaned == 0)),
        "phishing_samples": int(sum(y_cleaned == 1)),
        "safe_percentage": round(float(sum(y_cleaned == 0) / len(y_cleaned) * 100), 2),
        "phishing_percentage": round(float(sum(y_cleaned == 1) / len(y_cleaned) * 100), 2)
    },
    "evaluation_metrics": {
        "accuracy": round(float(df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'Accuracy'].values[0]), 6),
        "precision": round(float(df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'Precision'].values[0]), 6),
        "recall": round(float(df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'Recall'].values[0]), 6),
        "f1_score": round(float(df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'F1-Score'].values[0]), 6),
        "roc_auc": round(float(df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'ROC-AUC'].values[0]), 6)
    },
    "confusion_matrix": {
        "true_negatives": tn_b,
        "false_positives": fp_b,
        "false_negatives": fn_b,
        "true_positives": tp_b,
        "false_positive_rate": round(fp_b / (fp_b + tn_b), 6),
        "false_negative_rate": round(fn_b / (fn_b + tp_b), 6)
    }
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print("Saved Files:")
print(f"  1. Pipeline: {pipeline_path.resolve()} ({pipeline_path.stat().st_size / (1024 * 1024):.2f} MB)")
print(f"  2. Metadata: {metadata_path.resolve()} ({metadata_path.stat().st_size / 1024:.2f} KB)")


Saved Files:
  1. Pipeline: C:\Users\baps\OneDrive\Desktop\ai-scam-phishing-detector\ml\models\url_phishing_pipeline.joblib (0.41 MB)
  2. Metadata: C:\Users\baps\OneDrive\Desktop\ai-scam-phishing-detector\ml\models\url_phishing_metadata.json (1.95 KB)


## 14. Reload Verification

Reloading the serialized `.joblib` pipeline and verifying predictions on real held-out test data.

In [16]:
# Reload pipeline from disk
loaded_url_pipeline = joblib.load(pipeline_path)
with open(metadata_path, 'r', encoding='utf-8') as f:
    loaded_metadata = json.load(f)

print("Reload Verification: Artifacts successfully reloaded from disk.")

# Select a real test-set sample
test_sample_url = X_test_raw.iloc[0]
actual_label = int(y_test.iloc[0])

# Run inference
pred_label = int(loaded_url_pipeline.predict([test_sample_url])[0])
pred_probs = loaded_url_pipeline.predict_proba([test_sample_url])[0]
phish_confidence = float(pred_probs[1])

print("\n=== REAL TEST SAMPLE PREDICTION ===")
print(f"  URL:                 {test_sample_url}")
print(f"  Actual Label:        {actual_label} ({'Phishing / Scam' if actual_label == 1 else 'Safe / Legitimate'})")
print(f"  Pipeline Prediction: {pred_label} ({'Phishing / Scam' if pred_label == 1 else 'Safe / Legitimate'})")
print(f"  Phishing Confidence: {phish_confidence * 100:.2f}%")
print(f"  Status:              {'PASS (Prediction matches actual)' if pred_label == actual_label else 'Mismatch'}")

# Verify complete test set consistency
all_reloaded_preds = loaded_url_pipeline.predict(X_test_raw)
np.testing.assert_array_equal(all_reloaded_preds, y_pred_pipe)
print(f"\nFinal Consistency Check: All {len(X_test_raw):,} test-set predictions from the reloaded pipeline match in-memory model perfectly!")


Reload Verification: Artifacts successfully reloaded from disk.

=== REAL TEST SAMPLE PREDICTION ===
  URL:                 http://www.worldmedicsky.info
  Actual Label:        1 (Phishing / Scam)
  Pipeline Prediction: 1 (Phishing / Scam)
  Phishing Confidence: 100.00%
  Status:              PASS (Prediction matches actual)



Final Consistency Check: All 47,074 test-set predictions from the reloaded pipeline match in-memory model perfectly!


## 15. Final Summary

Consolidated summary of dataset, feature engineering, model benchmarks, and production deployment artifacts.

In [17]:
summary_df = pd.DataFrame({
    'Metric / Property': [
        'Dataset Name',
        'Raw Samples',
        'Cleaned Samples',
        'Train / Test Split',
        'Engineered Features',
        'Best Model',
        'Test Accuracy',
        'Test Precision',
        'Test Recall',
        'Test F1-Score',
        'Test ROC-AUC',
        'Pipeline Path',
        'Metadata Path',
        'Reload Verification'
    ],
    'Value': [
        'PhiUSIIL Phishing URL Dataset',
        '235,795 rows',
        '235,370 rows (425 duplicate URLs removed)',
        '188,296 train / 47,074 test (80/20 Stratified)',
        f"{len(FEATURE_NAMES)} lexical/structural URL features",
        'XGBoost Classifier (XGBClassifier)',
        f"{df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'Accuracy'].values[0]:.4f}",
        f"{df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'Precision'].values[0]:.4f}",
        f"{df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'Recall'].values[0]:.4f}",
        f"{df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'F1-Score'].values[0]:.4f}",
        f"{df_comparison.loc[df_comparison['Model'] == 'XGBoost', 'ROC-AUC'].values[0]:.4f}",
        str(pipeline_path),
        str(metadata_path),
        'PASSED'
    ]
})

print("=== URL PHISHING DETECTION WORKFLOW SUMMARY ===")
print(summary_df.to_string(index=False))


=== URL PHISHING DETECTION WORKFLOW SUMMARY ===
  Metric / Property                                                                                           Value
       Dataset Name                                                                   PhiUSIIL Phishing URL Dataset
        Raw Samples                                                                                    235,795 rows
    Cleaned Samples                                                       235,370 rows (425 duplicate URLs removed)
 Train / Test Split                                                  188,296 train / 47,074 test (80/20 Stratified)
Engineered Features                                                              23 lexical/structural URL features
         Best Model                                                              XGBoost Classifier (XGBClassifier)
      Test Accuracy                                                                                          0.9959
     Test Precision     